In [5]:
# !pip install torch_tensorrt

In [1]:
import torch
import torch_tensorrt
import timm
import time
import numpy as np
import torch.backends.cudnn as cudnn

[08/28/2026-10:32:40] [TRT] [W] Functionality provided through tensorrt.plugin module is experimental.


Unable to import quantization op. Please install modelopt library (https://github.com/NVIDIA/TensorRT-Model-Optimizer?tab=readme-ov-file#installation) to add support for compiling quantized models
Unable to import quantize op. Please install modelopt library (https://github.com/NVIDIA/TensorRT-Model-Optimizer?tab=readme-ov-file#installation) to add support for compiling quantized models


In [2]:
for i in range(torch.cuda.device_count()):
    print(f"GPU device: {torch.cuda.get_device_name(i)}")

GPU device: NVIDIA GB10


In [6]:
print(timm.list_models("*net", pretrained=True))

['dla60_res2net.in1k', 'test_byobnet.r160_in1k', 'test_efficientnet.r160_in1k', 'test_nfnet.r160_in1k', 'test_resnet.r160_in1k']


In [10]:
efficientnet_b0 = timm.create_model('efficientnet_b0',pretrained=True)

In [11]:
model = efficientnet_b0.eval().to("cuda")
detections_batch = model(torch.randn(128, 3, 224, 224).to("cuda"))
detections_batch.shape

torch.Size([128, 1000])

In [14]:
cudnn.benchmark = True

def benchmark(model, input_shape=(1024, 3, 512, 512), dtype='fp32', nwarmup=50, nruns=1000):
    input_data = torch.randn(input_shape)
    input_data = input_data.to("cuda")
    if dtype=='fp16':
        input_data = input_data.half()
        
    print("Warm up ...")
    with torch.no_grad():
        for _ in range(nwarmup):
            features = model(input_data)
    torch.cuda.synchronize()
    print("Start timing ...")
    timings = []
    with torch.no_grad():
        for i in range(1, nruns+1):
            start_time = time.time()
            pred_loc  = model(input_data)
            torch.cuda.synchronize()
            end_time = time.time()
            timings.append(end_time - start_time)
            if i%10==0:
                print('Iteration %d/%d, avg batch time %.2f ms'%(i, nruns, np.mean(timings)*1000))

    print("Input shape:", input_data.size())
    print('Average throughput: %.2f images/second'%(input_shape[0]/np.mean(timings)))

In [17]:
model = efficientnet_b0.eval().to("cuda")
benchmark(model, input_shape=(1, 3, 224, 224), nruns=100)

Warm up ...
Start timing ...
Iteration 10/100, avg batch time 1.86 ms
Iteration 20/100, avg batch time 1.86 ms
Iteration 30/100, avg batch time 1.86 ms
Iteration 40/100, avg batch time 1.85 ms
Iteration 50/100, avg batch time 1.85 ms
Iteration 60/100, avg batch time 1.88 ms
Iteration 70/100, avg batch time 1.89 ms
Iteration 80/100, avg batch time 1.89 ms
Iteration 90/100, avg batch time 1.89 ms
Iteration 100/100, avg batch time 1.89 ms
Input shape: torch.Size([1, 3, 224, 224])
Average throughput: 529.55 images/second


In [18]:
traced_model = torch.jit.trace(model, torch.randn((1,3,224,224)).to("cuda"))
torch.jit.save(traced_model, "efficientnet_b0_traced.jit.pt")
benchmark(traced_model, input_shape=(1, 3, 224, 224), nruns=100)

Warm up ...
Start timing ...
Iteration 10/100, avg batch time 1.35 ms
Iteration 20/100, avg batch time 1.39 ms
Iteration 30/100, avg batch time 1.39 ms
Iteration 40/100, avg batch time 1.39 ms
Iteration 50/100, avg batch time 1.38 ms
Iteration 60/100, avg batch time 1.39 ms
Iteration 70/100, avg batch time 1.38 ms
Iteration 80/100, avg batch time 1.38 ms
Iteration 90/100, avg batch time 1.38 ms
Iteration 100/100, avg batch time 1.37 ms
Input shape: torch.Size([1, 3, 224, 224])
Average throughput: 727.89 images/second


In [21]:
trt_model = torch_tensorrt.compile(model, 
    inputs= [torch_tensorrt.Input((1, 3, 224, 224))],
    enabled_precisions= {torch.float16} # Run with FP16
)

  return cls.__new__(cls, *args)




In [23]:
benchmark(trt_model, input_shape=(1, 3, 224, 224), nruns=100, dtype="float")

Warm up ...
Start timing ...
Iteration 10/100, avg batch time 0.92 ms
Iteration 20/100, avg batch time 0.97 ms
Iteration 30/100, avg batch time 0.95 ms
Iteration 40/100, avg batch time 0.95 ms
Iteration 50/100, avg batch time 0.94 ms
Iteration 60/100, avg batch time 0.95 ms
Iteration 70/100, avg batch time 0.95 ms
Iteration 80/100, avg batch time 0.94 ms
Iteration 90/100, avg batch time 0.94 ms
Iteration 100/100, avg batch time 0.94 ms
Input shape: torch.Size([1, 3, 224, 224])
Average throughput: 1063.48 images/second
